In [ ]:
import torch 
from torch.utils.data import Dataset, DataLoader
from pyteomics import mgf
import numpy as np
import logging
import torch.nn as nn
import torch
import torch.nn.functional as F
from torch import optim
import networkx as nx
from collections import deque
from matplotlib import pyplot as plt

from sklearn.metrics import (
    adjusted_mutual_info_score as AMI,
    adjusted_rand_score as ARI,
)
import random


In [ ]:
def get_bin_index(mz, min_mz, bin_size):
    relative_mz = mz - min_mz
    return max(0, int(np.floor(relative_mz / bin_size)))

def bin_spectrum(mz_array, intensity_array, max_mz=2500, min_mz=50.5, bin_size=1.0005079):
    """
    bin spectrum and this algorithm reference from 'https://github.com/dhmay/param-medic/blob/master/parammedic/binning.pyx'
    :param mz_array:
    :param intensity_array:
    :param max_mz:
    :param min_mz:
    :param bin_size:
    :return:
    """
    nbins = int(float(max_mz - min_mz) / float(bin_size)) + 1
    results = np.zeros(nbins)

    for index in range(len(mz_array)):
        mz = mz_array[index]
        intensity = intensity_array[index]
        intensity = np.sqrt(intensity)
        if mz < min_mz or mz > max_mz:
            continue
        bin_index = get_bin_index(mz, min_mz, bin_size)

        if bin_index < 0 or bin_index > nbins - 1:
            continue
        if results[bin_index] == 0:
            results[bin_index] = intensity
        else:
            results[bin_index] += intensity

    intensity_sum = results.sum()

    if intensity_sum > 0:
        results /= intensity_sum
        # spectrum_dict[key] = results
    else:
        logging.debug('zero intensity found')
    return results


# model input data preparation

In [ ]:
mgf_file_path = "./spectraList.mgf"    
spectra = list(mgf.read(mgf_file_path))


In [ ]:
traning_spectrum_data = []
for spectrum in spectra:
    # seperate to bins and
    quantized_spectra =bin_spectrum(spectrum["m/z array"], spectrum["intensity array"])
    traning_spectrum_data.append(quantized_spectra)
    

In [ ]:
# traning_spectrum_data

Creating the DBScan


In [ ]:
class DBSCANBase:
    def __init__(self, eps, minPts):
        self._eps = eps
        self._minPts = minPts

    def fit(self, X):
        pass

    def score(self, X, y):
        y_pred = self.predict(X)
        return (AMI(y, y_pred), ARI(y, y_pred))

    def predict(self, X):
        pass

In [ ]:
class DBSCANGraphBased(DBSCANBase):
    
    def __init__(self, eps, minPts):
        super().__init__(eps, minPts)
    def fit(self, X):
        self._construct_graph(X)
        self._get_clusters()
        self._get_centroids(X)
        self._cluster_outliers(X)

    def predict(self, X):
        y = []
        for x in X:
            distances = {
                label: self._distance(x, centroid)
                for label, centroid in self._centroids.items()
            }
            y.append(min(distances, key=distances.get))
        return y

    def _construct_graph(self, X):
        cardinality = len(X)
        graph = nx.Graph()
        graph.add_nodes_from(
            list(range(cardinality)), is_core=False, visited=False, cluster=None
        )
        for i in range(cardinality):
            for j in range(i + 1, cardinality):
                if self._distance(X[i], X[j]) < self._eps:
                    graph.add_edge(i, j)
            if graph.degree[i] >= self._minPts:
                graph.nodes[i]["is_core"] = True
        self._graph = graph

    def _get_clusters(self):
        cluster_label = 0
        for node in self._graph:
            if not self._graph.nodes[node]["visited"] and self._graph.nodes[node][
                "is_core"
            ]:
                self._graph.nodes[node]["visited"] = True
                self._graph.nodes[node]["cluster"] = cluster_label
                self._bfs_node(node, cluster_label)
                cluster_label += 1
        self._num_clusters = cluster_label

    def _bfs_node(self, source, cluster_label):
        neighbors = self._graph.neighbors
        queue = deque([(source, len(self._graph), neighbors(source))])
        while queue:
            _, depth_now, children = queue[0]
            try:
                child = next(children)
                if not self._graph.nodes[child]["visited"]:
                    self._graph.nodes[child]["visited"] = True
                    self._graph.nodes[child]["cluster"] = cluster_label
                    if depth_now > 1:
                        queue.append((child, depth_now - 1, neighbors(child)))
            except StopIteration:
                queue.popleft()

    def _get_centroids(self, X):
        self._centroids = {}
        for i in range(self._num_clusters):
            X_list = []
            for node in self._graph:
                if self._graph.nodes[node]["cluster"] == i:
                    X_list.append(X[i])
            self._centroids.update({i: np.mean(X_list, axis=0)})

    def _cluster_outliers(self, X):
        for node in self._graph:
            if self._graph.nodes[node]["cluster"] is None:
                self._graph.nodes[node]["cluster"] = self.predict(X[node])

    @staticmethod
    def _distance(p1, p2):
        return np.linalg.norm(p1 - p2)


In [ ]:
traning_spectrum_data = np.array(traning_spectrum_data)
clusterer = DBSCANGraphBased(0.4, 5)
clusterer.fit(traning_spectrum_data)
# ami, ari = clusterer.score(X, y)
# print(f'AMI: {ami}, ARI: {ari}')
y_pred = clusterer.predict(traning_spectrum_data)
best_centroids = np.array(
    [centroid for centroid in clusterer._centroids.values()]
)

plt.scatter(traning_spectrum_data[:,0], traning_spectrum_data[:,1], c=y_pred);
plt.scatter(best_centroids[:,0], best_centroids[:,1], marker='x', color='red');


In [ ]:
def pari_pectrum(spectra, label):
    spectra_count = len(spectra)
    labeled_spectral_pairs =[]
    for i in range(spectra_count):
        index1 = random.randint(0,spectra_count)
        index2 = random.randint(0,spectra_count)
        # if (label[index1]== label[index2]):
        #     is_same_label = True
        # else:
        #     is_same_label = False
        # labeled_spectral_pairs.append([spectra[index1], spectra[index2],is_same_label])
    # return labeled_spectral_pairs

pari_pectrum(traning_spectrum_data, y_pred)

# Loading the deconvoluted spectra using a dataloader

In [ ]:
class SpectrumDataset(Dataset):
    def __init__(self,mgf_file_path, max_peaks=200):
        self.mgf_file_path = mgf_file_path
        self.max_peaks = max_peaks
        self.spectra = list(mgf.read(mgf_file_path))
        
    def __len__(self):
        return len(self.spectra)

    def __getitem__(self, index):
        spectrum = self.spectra[index]
        mz_values = spectrum.get("m/z array", np.array([]))
        intensity_values = spectrum.get("intensity array", np.array([]))
        
        # normalize intensities
        intensity_values = intensity_values / np.max(intensity_values) if intensity_values.size else intensity_values

        return torch.tensor(np.stack(mz_values,intensity_values), axis=1)

    

In [ ]:
mgf_file_path = "./spectraList.mgf"
spectrum = list(mgf.read(mgf_file_path))    

In [ ]:
list(spectrum)[1].get("m/z array")

In [ ]:
mgf_file_path = "./spectraList.mgf"


dataset = SpectrumDataset(mgf_file_path)

Training_data_loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=1 )


In [ ]:
class SiamesNetwork(nn.Module):
    def __init__(self):
        super(SiamesNetwork, self).__init__()
        
        self.fc1 = nn.Sequential(
            nn.Linear(34,32),
            nn.SELU(),
            nn.Linear(32,5),
            nn.SELU()            
        )
        self.cnn11 = nn.Sequential(
            nn.Conv1d(1,30,3),
            nn.SELU(),
            nn.MaxPool1d(2),
            nn.SELU()
        )
        self.cnn21 = nn.Sequential(
            nn.Conv1d(1,30,3),
            nn.SELU(),
            nn.MaxPool1d(2),
            nn.SELU(),
            nn.Conv1d(30,30,3),
            nn.MaxPool1d(2),
            nn.SELU()
        )
        self.fc2 =nn.Linear(25775,32)
    
    def forward_once(self, x):
        # Get feature embeddings
        output = self.fc1(x)
        output = self.cnn11(x)
        output = self.cnn21(x)
        output = self.fc2(x)
        
        # Compute Euclidean distance
        return output
    
    def forward(self, x1, x2):
        output1 = self.forward_once(x1)
        output2 = self.forward_once(x2)
        return output1, output2

In [ ]:
# Define the Contrastive Loss Function
class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, output1, output2, label):
      # Calculate the euclidian distance and calculate the contrastive loss
      euclidean_distance = F.pairwise_distance(output1, output2, keepdim = True)

      loss_contrastive = torch.mean((1-label) * torch.pow(euclidean_distance, 2) +
                                    (label) * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2))


      return loss_contrastive

In [ ]:
net =SiamesNetwork()
lossFunc = ContrastiveLoss()
optimzer = optim.Adam(net.parameters(), lr = 0.0005)

In [ ]:
epoches = 10

for epoch in range(0, epoches):
    for i, data in enumerate(Training_data_loader,0):
        spectrum1, spectrum2, label = data
        optimzer.zero_grad()
        output1, output2 = net(spectrum1, spectrum2)
        loss_constrastive = lossFunc(output1, clear_output, label)
        loss_constrastive.backward()
        optimzer.step()
        
        if i % 10 == 0:
            print(f"Epoch {epoch+1}/{100}, Iteration {i}, Loss: {loss_constrastive.item()}")
 
        